In [1]:
import pandas as pd
from tqdm import tqdm

from src.config import SAMPLE_REVIEWS_PATH
from src.formatacao import limpar_texto_simbolico, tokenizar_texto, extrair_frequencia, limpeza_texto_com_acento
from src.ngramas import extrair_ngramas
from src.extracao_gramatica import aplicar_pos_tagging
from src.polaridade import calcular_polaridade_texto



df_sample = pd.read_parquet(SAMPLE_REVIEWS_PATH)
df_sample.head()
"""
Para etapa de solução simbólica, vamos focar nas colunas 'review' e 'score'.
foi selecionado um sample de 50000 reviews para facilitar a análise e o desennvolvimento
nessa etapa

df_sample tem as seguintes colunas:
- review: texto de revisão do produto
- score: nota dada pelo usuário (1 a 5)
- product_name: nome do produto (muitos valores nulos)
- source: fonte da revisão

"""

"\nPara etapa de solução simbólica, vamos focar nas colunas 'review' e 'score'.\nfoi selecionado um sample de 50000 reviews para facilitar a análise e o desennvolvimento\nnessa etapa\n\ndf_sample tem as seguintes colunas:\n- review: texto de revisão do produto\n- score: nota dada pelo usuário (1 a 5)\n- product_name: nome do produto (muitos valores nulos)\n- source: fonte da revisão\n\n"

In [2]:
# product_name tem muitos valores nulos, vai ser desconsiderado para essa análise
print(df_sample['product_name'].isnull().sum())

df_sample['score'].value_counts()

28178


score
4    10000
1    10000
2    10000
5    10000
3    10000
Name: count, dtype: int64

In [3]:
# aplicando a limpeza e tokenização dos dados:

df_sample['review_limpa'] = df_sample['review'].apply(limpar_texto_simbolico)

df_sample['tokens'] = df_sample['review_limpa'].apply(lambda x: tokenizar_texto(x))

print(df_sample[['review_limpa', 'tokens']].head())

                                        review_limpa  \
0  PRODUTO EXCELENTE, MEU AFILHADO AMOU E O DESIG...   
1  Veio com o reservatorio descolado e vaza toda ...   
2  Fraco, produto bem mais ou menos e deixou a de...   
3  O que dizer de um produto que apos instalado, ...   
4  entrega super rapida ,adorei o produto super o...   

                                              tokens  
0  [{'palavra': 'excelente', 'e_maiusculo': True,...  
1  [{'palavra': 'veio', 'e_maiusculo': False, 'te...  
2  [{'palavra': 'fraco', 'e_maiusculo': False, 't...  
3  [{'palavra': 'dizer', 'e_maiusculo': False, 't...  
4  [{'palavra': 'entrega', 'e_maiusculo': False, ...  


In [4]:
len(df_sample)

50000

In [5]:
df_sample['review_limpa'].head()

0    PRODUTO EXCELENTE, MEU AFILHADO AMOU E O DESIG...
1    Veio com o reservatorio descolado e vaza toda ...
2    Fraco, produto bem mais ou menos e deixou a de...
3    O que dizer de um produto que apos instalado, ...
4    entrega super rapida ,adorei o produto super o...
Name: review_limpa, dtype: str

In [6]:
df_sample['tokens'].head()

0    [{'palavra': 'excelente', 'e_maiusculo': True,...
1    [{'palavra': 'veio', 'e_maiusculo': False, 'te...
2    [{'palavra': 'fraco', 'e_maiusculo': False, 't...
3    [{'palavra': 'dizer', 'e_maiusculo': False, 't...
4    [{'palavra': 'entrega', 'e_maiusculo': False, ...
Name: tokens, dtype: object

In [7]:
# visualizando a frequência das palavras mais comuns:

df_frequencia = extrair_frequencia(df_sample['tokens'])

print(df_frequencia.head(50))

       palavra  frequencia
0          nao       48197
1       gostei       21804
2        muito       19589
3          bom       10228
4         mais        9655
5          bem        5965
6    qualidade        5813
7    recomendo        5625
8      comprei        4968
9           so        4715
10         boa        4556
11     entrega        4313
12        veio        4257
13      cabelo        4181
14       otimo        3959
15       preco        3715
16        nada        3623
17   excelente        3556
18      recebi        3475
19    aparelho        3319
20      chegou        3159
21       prazo        3110
22      compra        2992
23       pouco        2875
24       porem        2864
25          tv        2720
26     comprar        2694
27      imagem        2651
28      melhor        2577
29       facil        2446
30        dias        2280
31  americanas        2276
32         nem        2270
33       super        2186
34       custo        2164
35     bateria        2156
3

In [8]:
# visualizando os n-gramas mais comuns:

df_ngramas = extrair_ngramas(df_sample['tokens'], n=2)
print(df_ngramas.head(50))

print("\n")

df_ngramas_trigramas = extrair_ngramas(df_sample['tokens'], n=3)
print(df_ngramas_trigramas.head(50))

              expressao  frequencia
0            nao gostei       10197
1             muito bom        3998
2            gostei nao        2011
3            nao recebi        1836
4       custo beneficio        1754
5          gostei muito        1713
6         nao recomendo        1529
7             muito boa        1040
8         boa qualidade         816
9   assistencia tecnica         701
10            muito bem         701
11              nao sei         676
12            vale pena         676
13      otima qualidade         661
14              bom nao         658
15          gostei nada         657
16         nao funciona         655
17         gostei preco         610
18             nao veio         578
19         deixa cabelo         573
20           nao chegou         565
21     qualidade imagem         540
22  minhas expectativas         537
23           muito ruim         534
24      super recomendo         519
25         chegou prazo         517
26        prazo entrega     

In [9]:
# Reviews com acentos para pos tagging:
df_review_limpa_com_acento = df_sample['review'].apply(limpeza_texto_com_acento)


df_classes_gramaticais = df_review_limpa_com_acento.apply(aplicar_pos_tagging)

In [10]:
print(df_classes_gramaticais.head(10))



0    [{'palavra': 'produto', 'classe': 'PROPN'}, {'...
1    [{'palavra': 'veio', 'classe': 'VERB'}, {'pala...
2    [{'palavra': 'fraco', 'classe': 'PROPN'}, {'pa...
3    [{'palavra': 'o', 'classe': 'PRON'}, {'palavra...
4    [{'palavra': 'entrega', 'classe': 'VERB'}, {'p...
5    [{'palavra': 'gostei', 'classe': 'VERB'}, {'pa...
6    [{'palavra': 'sei', 'classe': 'VERB'}, {'palav...
7    [{'palavra': 'fiquei', 'classe': 'VERB'}, {'pa...
8    [{'palavra': 'realizei', 'classe': 'VERB'}, {'...
9    [{'palavra': 'só', 'classe': 'ADV'}, {'palavra...
Name: review, dtype: object


In [11]:
tqdm.pandas()
# aplicando a função de polaridade para cada review limpa
df_sample['resultado_dict'] = df_sample['review_limpa'].progress_apply(calcular_polaridade_texto)
# transformando a coluna resultado_dict em colunas separadas
df_resultados = df_sample['resultado_dict'].apply(pd.Series)
# concatenando os resultados com o dataframe original
df_final = pd.concat([df_sample, df_resultados], axis=1)

df_final = df_final.drop(columns=['resultado_dict'])

100%|██████████| 50000/50000 [08:35<00:00, 96.93it/s] 


In [22]:
print(df_final['review_limpa'][2])

df_final['opinioes'][2]



Fraco, produto bem mais ou menos e deixou a desejar


[{'expressao': 'bem mais ou menos',
  'nota': -1.0,
  'categoria': 'Negativo',
  'regra': 'OPINIAO_MAIS_OU_MENOS'},
 {'expressao': 'deixou a desejar',
  'nota': -2.0,
  'categoria': 'Negativo',
  'regra': 'OPINIAO_DEIXOU_A_DESEJAR'}]